1. Выбор датасета:

Выбор пал на предложенный авторами задания датасет - https://www.kaggle.com/code/aryantiwari123/iris-species/input

Колонки выбранного датасета:
- Id (айди)
- SepalLengthCm (длина "чашелистика")
- SepalWidthCm (ширина "чашелистица")
- PetalLengthCm (длина лепестка)
- PetalWidthCm (ширина лепестка)
- Species (вид)

Выбрал его, потому что:
- он был первый в списке предложенных для обработки
- ну, собственно, и всё

2. Первичный анализ данных датасета Iris Species:

In [195]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

sns.set_style('dark')

df = pd.read_csv('Iris.csv')

In [196]:
df.head(3)

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa


In [197]:
df.tail(3)

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
147,148,6.5,3.0,5.2,2.0,Iris-virginica
148,149,6.2,3.4,5.4,2.3,Iris-virginica
149,150,5.9,3.0,5.1,1.8,Iris-virginica


In [198]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             150 non-null    int64  
 1   SepalLengthCm  150 non-null    float64
 2   SepalWidthCm   150 non-null    float64
 3   PetalLengthCm  150 non-null    float64
 4   PetalWidthCm   150 non-null    float64
 5   Species        150 non-null    str    
dtypes: float64(4), int64(1), str(1)
memory usage: 7.2 KB


In [199]:
df.describe()

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm
count,150.000000,150.000000,150.000000,150.000000,150.000000
mean,75.500000,5.843333,3.054000,3.758667,1.198667
std,43.445368,0.828066,0.433594,1.764420,0.763161
min,1.000000,4.300000,2.000000,1.000000,0.100000
25%,38.250000,5.100000,2.800000,1.600000,0.300000
50%,75.500000,5.800000,3.000000,4.350000,1.300000
75%,112.750000,6.400000,3.300000,5.100000,1.800000
max,150.000000,7.900000,4.400000,6.900000,2.500000


In [200]:
df.select_dtypes(include=['int64', 'float64']).mean()

Id               75.500000
SepalLengthCm     5.843333
SepalWidthCm      3.054000
PetalLengthCm     3.758667
PetalWidthCm      1.198667
dtype: float64

In [201]:
df.select_dtypes(include=['int64', 'float64']).median()

Id               75.50
SepalLengthCm     5.80
SepalWidthCm      3.00
PetalLengthCm     4.35
PetalWidthCm      1.30
dtype: float64

In [202]:
df.isnull().sum()

Id               0
SepalLengthCm    0
SepalWidthCm     0
PetalLengthCm    0
PetalWidthCm     0
Species          0
dtype: int64

In [203]:
df.duplicated().sum()

np.int64(0)

In [204]:
df.shape

(150, 6)

In [205]:
df.select_dtypes(include=['int64', 'float64']).var()

Id               1887.500000
SepalLengthCm       0.685694
SepalWidthCm        0.188004
PetalLengthCm       3.113179
PetalWidthCm        0.582414
dtype: float64

In [206]:
df.select_dtypes(include=['int64', 'float64']).skew()

Id               0.000000
SepalLengthCm    0.314911
SepalWidthCm     0.334053
PetalLengthCm   -0.274464
PetalWidthCm    -0.104997
dtype: float64

In [207]:
df.select_dtypes(include=['int64', 'float64']).kurtosis()

Id              -1.200000
SepalLengthCm   -0.552064
SepalWidthCm     0.290781
PetalLengthCm   -1.401921
PetalWidthCm    -1.339754
dtype: float64

In [208]:
df['Species'].value_counts(normalize=True)

Species
Iris-setosa        0.333333
Iris-versicolor    0.333333
Iris-virginica     0.333333
Name: proportion, dtype: float64

In [209]:
df_melted = df.melt(var_name='Attribute', value_name='Value')
fig = px.box(
    df_melted,
    x='Attribute',
    y='Value',
    template='plotly_dark'
)

fig.show()

In [210]:
df_cols = df.select_dtypes(include=['float64', 'int64']).columns.to_list()
for i in df_cols:
    fig = px.histogram(
        df,
        x=i,
        height=500,
        width=800,
        nbins=50,
    )
    fig.update_layout(
        yaxis_title='Frequency'
    )
    fig.update_traces(
        marker_line_color='black', 
        marker_line_width=1.5
    )
    fig.show()

Исходя из проведённого анализа, можно сделать следующие выводы:
- датасет маленький (150 строк на 6 колонок);
- выбросов практически нет;
- нет "отсутствующих" значений, пропуски заполнять не нужно;
- единственная текстовая колонка - название вида (Species), исходя из этого именно она будет являться target переменной;
- распределение классов target переменной идеально равномерно (0.(3) для каждого класса);
- исходя из предложенных выше графиков, ряд признаков сильно соответствует нормальному распределению (исключение - SepalWidthCm).Наблюдается как мультимодальность (2 и более пика), так и реже встречающееся среднее значение (отрицательный эксцесс у ряда признаков);

Вывод: единственная проблема датсета - его размер. В случае с KNN 150 объектов мало для уверенности в получении верно предсказывающей результат модели.

3. Подготовка данных:

Масштабирование признаков. В случае с KNN зачастую используют стандартное отклонение (в sklearn это StandartScaler). 

Нормализовать данные нужно, чтобы одни признаки не влияли на выбор ответа обучаемой моделью сильнее, чем другие (если один признак меняется в диапазоне [0,1], а другой [0,100], на расстояние второй будет влиять сильнее при достаточном изменение показателя (изменение на 3 пункта, что есть 3% от всего диапазона значений второго признака влияют КАК МИНИМУМ в 3 раза сильнее на итоговый ответ модели, чем первый признак на ВСЁМ диапазоне им принимаемых значений)). Нормализация данных позволяет минимизировать "влияние" подобных отклонений на итоговый ответ, оставив лишь реальное "влияние" признаков.

Ох я и намудрил...)))

Теперь определим target:
- интуитивно гораздо правильнее угадывать не длину или ширину листа (хотя это было бы интересно...), а название вида;
- единственный категориальный признак, не представленный числом в данный момент, является название вида;

Вывод: target переменная - Species

Разделение на train и test. 

Так как у нас 3 класса, каждый занимает в исходном датасете ровно треть, логично поделить датасет следующим образом: 70% выборка на тренировку - 30% тест (в противном случае, если на тест оставить меньше значений, результат будет сильно коррелироваться в зависимости от исходных данных (если повезёт, может и всё угадаем, а может и, наоборот, точность будет слишком низкой)). Далее - нужно сделать равное разбиение по классам (чтобы в train попали все 3 класса, не получится просто взять первые 70% строк в train, сделаем хитрее).

Путём недолгих подсчётов:
В каждом классе 50 объектов, каждый класс представлен в test выборке 50/100*30 = 15-ю объектами. Так формируем test, остальное - train выборку.

На тестовой выборке подбор параметров осуществлять не будем, так как в таком случае есть вероятность, что модель просто "запомнит", какому набору входных данных какой вид соответствует, но при этом сама не научится решать задачу.

Кодирование данных.

Наконец, кодирование данных. Так как в нашем датасете всего 1 категориальный признак нуждается в кодировке, но именно он и является target переменной, мы не будем кодировать данные при поставленной задаче и данном датасете.

In [211]:
Y = df['Species']
X = df.drop(['Species', 'Id'], axis=1)

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.3,
    random_state=42,
    stratify=Y
)


In [212]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

После разбиения и стандартизации данных посмотрим, что получили:

In [213]:
X_train_scaled.mean(axis=0)

array([ 3.00288894e-16, -1.16983143e-15, -4.56777473e-16,  1.01506105e-16])

In [214]:
X_test_scaled.mean(axis=0)

array([-0.11643861,  0.02599252, -0.05081417, -0.02621151])

Очень близко к нулю на train, так как на нём всё считали, на тесте погрешность немного больше, но тоже около нуля.

In [215]:
X_train_scaled.std(axis=0)

array([1., 1., 1., 1.])

In [216]:
X_test_scaled.std(axis=0)

array([0.85754489, 0.8452672 , 0.9691546 , 0.93583444])

На train std=1, на test к нему стремится - следовательно всё замечательно.

4. Обучение KNN;
5. Подбор гиперпараметров;

Наконец, перейдём к самому интересному: обучение KNN и подбору наилучшего набора гиперпараметров. Так как наша модель достаточно мала, а знаний в интуитивном (и, конечно же, не только интуитивном) подборе гиперпараметров у меня недостаточно, чтобы сделать это легко и просто, мы переберём все возможные варианты (в дальнейшем, конечно, такую практику будем избегать, но сейчас, в рамках обучения и в качестве улучшения насмотренности, мы это сделаем) гиперпараметры. Кроме того, используем кросс-валидацию, дабы получить необходимый список наборов гиперпараметров, не прибегая к использования тестовых данных, что привело бы к нежелательному переобучению модели.

Для этого воспользуемся GridSearchCv:

In [176]:
param_grid = {
    'n_neighbors': range(1,20),
    'metric': ['euclidean', 'manhattan', 'chebyshev'],
    'weights': ['uniform', 'distance']
}

grid_search = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,
    scoring='accuracy',
    verbose=1
)

grid_search.fit(X_train_scaled, Y_train)

cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results = cv_results.sort_values('rank_test_score')

cv_results.head(10)

Fitting 5 folds for each of 114 candidates, totalling 570 fits


,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_metric,param_n_neighbors,param_weights,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
26,0.000658,0.000078,0.001419,0.000281,euclidean,14,uniform,"{'metric': 'euclidean', 'n_neighbors': 14, 'we...",1.000000,0.904762,1.0,1.000000,1.0,0.980952,0.038095,1
21,0.000673,0.000094,0.001041,0.000087,euclidean,11,distance,"{'metric': 'euclidean', 'n_neighbors': 11, 'we...",0.952381,0.952381,1.0,0.952381,1.0,0.971429,0.023328,2
27,0.000951,0.000266,0.001344,0.000221,euclidean,14,distance,"{'metric': 'euclidean', 'n_neighbors': 14, 'we...",0.952381,0.952381,1.0,0.952381,1.0,0.971429,0.023328,2
25,0.000615,0.000049,0.000891,0.000062,euclidean,13,distance,"{'metric': 'euclidean', 'n_neighbors': 13, 'we...",0.952381,0.952381,1.0,0.952381,1.0,0.971429,0.023328,2
19,0.000617,0.000091,0.000896,0.000079,euclidean,10,distance,"{'metric': 'euclidean', 'n_neighbors': 10, 'we...",0.952381,0.952381,1.0,0.952381,1.0,0.971429,0.023328,2
23,0.000783,0.000231,0.001232,0.000408,euclidean,12,distance,"{'metric': 'euclidean', 'n_neighbors': 12, 'we...",0.952381,0.952381,1.0,0.952381,1.0,0.971429,0.023328,2
17,0.000617,0.000055,0.000999,0.000209,euclidean,9,distance,"{'metric': 'euclidean', 'n_neighbors': 9, 'wei...",0.952381,0.952381,1.0,0.952381,1.0,0.971429,0.023328,2
29,0.000709,0.000041,0.001045,0.000095,euclidean,15,distance,"{'metric': 'euclidean', 'n_neighbors': 15, 'we...",0.952381,0.952381,1.0,0.952381,1.0,0.971429,0.023328,2
31,0.000702,0.000095,0.000979,0.000077,euclidean,16,distance,"{'metric': 'euclidean', 'n_neighbors': 16, 'we...",0.952381,0.952381,1.0,0.952381,1.0,0.971429,0.023328,2
49,0.000701,0.000120,0.000918,0.000049,manhattan,6,distance,"{'metric': 'manhattan', 'n_neighbors': 6, 'wei...",0.952381,0.952381,1.0,0.952381,1.0,0.971429,0.023328,2


In [164]:
cv_results.tail(5)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_metric,param_n_neighbors,param_weights,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
78,0.000638,0.000075,0.001170,0.000084,chebyshev,2,uniform,"{'metric': 'chebyshev', 'n_neighbors': 2, 'wei...",0.952381,0.904762,0.904762,0.857143,0.857143,0.895238,0.035635,110
108,0.000607,0.000031,0.001272,0.000062,chebyshev,17,uniform,"{'metric': 'chebyshev', 'n_neighbors': 17, 'we...",0.952381,0.857143,0.761905,1.000000,0.904762,0.895238,0.081927,110
106,0.000594,0.000025,0.001242,0.000048,chebyshev,16,uniform,"{'metric': 'chebyshev', 'n_neighbors': 16, 'we...",0.952381,0.857143,0.809524,0.904762,0.904762,0.885714,0.048562,112
110,0.000624,0.000039,0.001307,0.000035,chebyshev,18,uniform,"{'metric': 'chebyshev', 'n_neighbors': 18, 'we...",0.952381,0.857143,0.761905,0.904762,0.904762,0.876190,0.064594,113
112,0.000601,0.000041,0.001255,0.000050,chebyshev,19,uniform,"{'metric': 'chebyshev', 'n_neighbors': 19, 'we...",0.952381,0.809524,0.761905,0.904762,0.857143,0.857143,0.067344,114


In [162]:
fig = px.scatter(
    cv_results, 
    x='param_n_neighbors',
    y='mean_test_score',
    color='rank_test_score'
)
fig.show()

Подметим некоторые важные моменты, после чего перейдём к работе с тестовыми данными (запуск test с подобранными выше параметрами) и, затем, к выводам о проделанной работе.

1. Чебышев и Манхэттен не справились! Почему? 

- Манхэттеновская метрика используется в случае, когда нужно быть устойчивее к выбросам данных, работая по принципу: как из точки A в точку B пройти либо строго по вертикали, либо строго по горизонтали, диагональ - нельзя;
- Метрика Чебышева же вообще фокусируется на выбросах, невелируя значимость средних значений;

Так как у нас фактически нет выбросов данных ввиду хорошо подобранного датасета, Евклидова метрика, меряющая, в сущности, обычное расстояние между точками, дала наиболее точный результат.

2. Пусть KNN и достаточна, чтобы практически со стопроцентной точностью предсказать target класс, скорее всего это произошло из-за лёгкого в обработке датасета: всего 3 target класса, сильно различающихся между собой;

3. Как ни странно (хотя, быть может, это и нормально), наиболее точной оказалась модель аж с 19-ю соседями в рассмотрении. Данный феномен не совсем понятно почему появился и требует более длительного осмысления. Мною ожидалось, что 7+-2 соседа - идеал для данной модели (в целом, на графике видно, что примерно в этом диапазоне (на самом деле этот диапазон чуть смещён - [8,12]) как раз-таки, даже при учёте различности других гиперпараметров, модель ведёт себя стабильнее всего);

4. Что ещё более удивительно, в топ-10 по точности лучший набор гиперпараметров ЕДИНСТВЕННЫЙ, использующий метрику весов uniform (остальные все зависят от дистанции);

5. Показатель std_test_score, говорящий о том, что выбранная модель не столь стабильна, сколь того хочется, в 2 раза выше у занявшей по точности первое место модели, нежели чем у моделей, занявших второе место;

Вывод из сделанных наблюдений: так как результаты, к огромному удивлению, оказались не совсем тривиальны, на тестовых данных мы запустим 2 набора гиперпараметров, а именно:
- 14 neighbors, euclidean, uniform
- 11 neighbors, euclidean, distance 

In [179]:
knn1 = KNeighborsClassifier(n_neighbors=14, weights='uniform', metric='euclidean')
knn2 = KNeighborsClassifier(n_neighbors=11, weights='distance', metric='euclidean')

knn1.fit(X_train_scaled, Y_train)
knn2.fit(X_train_scaled, Y_train)

knn1_predict = knn1.predict(X_test_scaled)
knn2_predict = knn2.predict(X_test_scaled)

In [191]:
print(f"KNN1 accuracy: {accuracy_score(Y_test, knn1_predict)}")

KNN1 accuracy: 0.9555555555555556


In [192]:
print("\nKNN1:\n", classification_report(Y_test, knn1_predict))


KNN1:
                  precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        15
Iris-versicolor       0.88      1.00      0.94        15
 Iris-virginica       1.00      0.87      0.93        15

       accuracy                           0.96        45
      macro avg       0.96      0.96      0.96        45
   weighted avg       0.96      0.96      0.96        45



In [193]:
print(f"KNN2 accuracy: {accuracy_score(Y_test, knn2_predict)}")

KNN2 accuracy: 0.9333333333333333


In [194]:
print("KNN2:\n", classification_report(Y_test, knn2_predict))

KNN2:
                  precision    recall  f1-score   support

    Iris-setosa       1.00      1.00      1.00        15
Iris-versicolor       0.83      1.00      0.91        15
 Iris-virginica       1.00      0.80      0.89        15

       accuracy                           0.93        45
      macro avg       0.94      0.93      0.93        45
   weighted avg       0.94      0.93      0.93        45



На удивление, более точной по ВСЕМ предложенным показателям оказалась именно первая модель...

Выводы:
- масштабирование в разы повысило точность модели, превратив её из бесполезного набора мат. функций в нечто, что уже почти способно (во всяком случае, справляется с этим явно лучше человека) предсказывать вид ириса;
- при количестве соседей от 10 до 15 модель выдаёт наиболее точные результаты. При меньшем количестве, опираясь на меньшее число k соседей точность ниже из-за высокой (стопроцентной) вероятности нелинейной границы между классами. При большем количестве, ввиду, быть может малого размера датасета, быть может ввиду других, на данный момент не установленных причин, точность также падает (скорее всего, верен именно первый вариант);
- Еклидова метрика при поставленной задаче - канон, так как датасет наш без выбросов и жёстко привязан к среднему значению;
- Веса: чаще в топе лучших моделей (9 из 10) встречалась зависимость веса точки от расстояния (distance), с другой стороны, наиболее точной оказалось именно модель с весами, равными 1. Видимо, в поставленной задаче классификации в общем случае всё стоит отдавать предпочтение именно distance;
- KNN идеально подходит под поставленную задачу, так как сам по себе датасет маленький, так как классов всего 3, так как признаков 6, дак ещё 1 из них - это Id, от которого спешно было принято решение избавиться, так как не было (кроме target переменной) категориальных признаков. В иных случаях можно присмотреться к таким методам обучения модели, как линейная регрессия (если уместно, с Lasso или Ridge), градиентный спуск и т.д.;
- итоговый набор гиперпараметров: [n_neighbors=14, weights='uniform', metric='euclidean'], точность на тестовой выборке - 95,(5)%;